# CPSMA 4313 Spring 2026 - Project: Create Dataset

**Name:** Prinsa Ghimire  
**Course Name:** Data Processing Visualization (CPSMA-4313-01)  
**Student Number:** 0374272  
**Instructor:** Dr. Nicholas Jacob

In [56]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

In [57]:
base_url = "https://www.ecok.edu"
main_url = "https://www.ecok.edu/housing/housing-options/"

headers = {"User-Agent": "Mozilla/5.0"}

In [58]:
response = requests.get(main_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    
    if "/housing/housing-options/" in href and href.endswith(".php"):
        full_url = urljoin(base_url, href)
        links.append(full_url)

links = list(set(links))
print("Housing links:", links)

Housing links: ['https://www.ecok.edu/housing/housing-options/stadium-apartments.php', 'https://www.ecok.edu/housing/housing-options/chokka-chaffa-hall.php', 'https://www.ecok.edu/housing/housing-options/pontotoc-hall.php', 'https://www.ecok.edu/housing/housing-options/tiger-commons.php', 'https://www.ecok.edu/housing/housing-options/briles-hall.php', 'https://www.ecok.edu/housing/housing-options/pesagi-hall.php']


In [59]:
ATTRIBUTES = [
    "Type of Hall", "Security", "Cable", "Eligibility",
    "Meal Plan", "WI-FI", "Proximity", "Communal Lobby",
    "Smoke & Tobacco Free", "Bathroom", "Free Laundry", "Elevator"
]

In [60]:
housing_data = []

for url in links:
    try:
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        # Get housing name
        title = soup.find("h1").get_text(strip=True)

        # Initialize all attributes
        row_data = {attr: "" for attr in ATTRIBUTES}

        table = soup.find("table")

        if table:
            cells = table.find_all("td")

            for cell in cells:
                strong_tag = cell.find("strong")

                if strong_tag:
                    key = strong_tag.get_text(strip=True)

                    # normalize key (important for matching)
                    key = key.replace("Wi-Fi", "WI-FI").strip()

                    # extract all <p> tags
                    p_tags = cell.find_all("p")

                    value = ""

                    if len(p_tags) > 1:
                        # extract text with separator for <br>
                        raw_value = p_tags[1].get_text(separator="|", strip=True)

                        # convert to comma separated
                        parts = [v.strip() for v in raw_value.split("|") if v.strip()]
                        value = ", ".join(parts)

                    # store only required attributes
                    if key in row_data:
                        row_data[key] = value if value else "Not Available"

        # build final row
        data_row = {
            "Housing_Name": title,
            "Source_URL": url
        }

        data_row.update(row_data)

        housing_data.append(data_row)

        time.sleep(1)

    except Exception as e:
        print("Error:", url, e)

In [61]:
df_housing = pd.DataFrame(housing_data)
df_housing.head(len(df_housing))

,Housing_Name,Source_URL,Type of Hall,Security,Cable,Eligibility,Meal Plan,WI-FI,Proximity,Communal Lobby,Smoke & Tobacco Free,Bathroom,Free Laundry,Elevator
0,Stadium Apartments,https://www.ecok.edu/housing/housing-options/s...,"Apartments, Two Bedroom","Resident Assistant on-call, Residence Director...",,Upperclassmen,Flex meal plan required,Not Available,Koi Ishto Stadium,Not Available,Not Available,One bathroom in each apartment,Laundy Room On-Site,Not Available
1,Chokka-Chaffa’ Hall,https://www.ecok.edu/housing/housing-options/c...,"Co-Ed, Double Occupancy","I.D. card access, Resident Assistant on-call, ...",Not Available,"Undergraduate students, Enrolled in 12+ credit...",Required,Not Available,University Center Bookstore,Not Available,Not Available,En-Room,Available on 2nd and 3rd floors,Not Available
2,Pontotoc Hall,https://www.ecok.edu/housing/housing-options/p...,"Co-Ed, Single and Double Occupancy","I.D. card access, Resident Assistant on-call, ...",Not Available,"Undergraduate students, Enrolled in 12+ credit...",Required,Not Available,"Taff Dining Hall, Linscheid Library, Koi Ishto...",Not Available,Not Available,Suite-style,Available on first floor of every wing,
3,Tiger Commons,https://www.ecok.edu/housing/housing-options/t...,"Co-Ed, Two Bedroom, Four Bedroom","Resident Assistant on-call, Residence Director...",,Upperclassmen,Flex Meal Plan required,Not Available,Linscheid Library,Located in Pesagi Hall,Not Available,"Two Bathroom, One Bathroom",Located on first floor of Pesagi Hall.,Not Available
4,Briles Hall,https://www.ecok.edu/housing/housing-options/b...,"Co-Ed, Single, Double and Triple Occupancy","I.D. card access, Resident Assistant on-call, ...",Not Available,Not Available,Not Available,Not Available,"University Center, Bookstore, Taff Cafeteria, ...",Not Available,Not Available,Suite-style,,
5,Pesagi Hall,https://www.ecok.edu/housing/housing-options/p...,"Co-Ed, Single and Double Occupancy","I.D. card access, Resident Assistant on-call, ...",Not Available,"Undergraduate students, Enrolled in 12+ credit...",Required,Not Available,Linsheid Library,Not Available,Not Available,Suite-style,Available on 1st floor,


In [62]:
# Get the page and parse
response = requests.get(rates_url)
soup = BeautifulSoup(response.text, "html.parser")

housing_data = []
current_hall = None

# Select the first table
table = soup.select("table")[0]

# Iterate rows
for row in table.select("tr"):
    cols = row.find_all("td")

    if len(cols) < 2:
        continue

    col1 = cols[0].get_text(strip=True)
    col2 = cols[1].get_text(strip=True)

    # Detect hall name
    if cols[0].find("strong") and not col2:
        current_hall = col1
        continue

    if not col1 or not col2:
        continue

    if cols[0].find("strong") and "Per Semester" in col2:
        current_hall = col1
        continue
    
    housing_data.append({
        "Hall": current_hall,
        "Room_Type": col1,
        "Price_Per_Semester": col2
    })

In [63]:

# Convert to DataFrame
df_housing_rates = pd.DataFrame(housing_data)
df_housing_rates.head(len(df_housing_rates))


,Hall,Room_Type,Price_Per_Semester
0,Briles Hall,Single - Private,"$2,545.00"
1,Briles Hall,Double,"$1,730.00"
2,Chokka Chaffa',Double,"$2,390.00"
3,Pesagi,Double,"$1,375.00"
4,Pontotoc,Single,"$2,225.00"
5,Pontotoc,Single - Private,"$2,450.00"
6,Pontotoc,Double,"$1,625.00"
7,Stadium Apartments,1 Bedroom,"$3,475.00"
8,Stadium Apartments,2 Bedroom,"$3,100.00"
9,Tiger Commons,2 Bedroom Suite,"$3,225.00"


In [64]:
def clean_hall_name(name):
    if pd.isna(name):
        return name
    s = str(name).lower().replace(" hall", "").strip()
    for ch in ("\u2019", "\u2018", "`"):
        s = s.replace(ch, "'")
    s = s.replace("-", " ")
    return " ".join(s.split())

df_housing_rates.columns = df_housing_rates.columns.str.strip()

df_rates = df_housing_rates.assign(Hall_clean=lambda d: d["Hall"].apply(clean_hall_name))
df_detail = df_housing.assign(Hall_clean=lambda d: d["Housing_Name"].apply(clean_hall_name))

final_df = pd.merge(df_rates, df_detail, on="Hall_clean", how="left").drop(
    columns=["Hall_clean"]
)

lead_cols = ["Hall", "Housing_Name", "Source_URL", "Room_Type", "Price_Per_Semester"]
ordered = [c for c in lead_cols if c in final_df.columns] + [
    c for c in final_df.columns if c not in lead_cols
]
final_df = final_df[ordered]

final_df.to_csv("final_ecu_housing_dataset.csv", index=False)